# talk

> A walk as a talk for a group: the deck shape, its checks, and a talk made from a walk.

A presenter can give a walk to a group. The talk is a list of slides. Each slide shows a part of the walk, one beat at a time,
and it can have one question for the room. This module is the engine of a talk: the shape of a deck, the checks of that shape,
the questions, a talk made from a walk, and the store calls. It imports no web code, so Leela can use the same shape and checks.

In [ ]:
#| default_exp talk

In [ ]:
#| hide
import os, tempfile
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

The imports are light: the store, the journey, the demos, the model calls, the prompts and the settings.

In [ ]:
#| export
import re, json, math, time, uuid
from fastcore.utils import *
from abhyasa.store import Learning, slugify, _src
from abhyasa.journey import minutes
from abhyasa.llm import ask_json, for_task
from abhyasa.prompts import register, prompt
from abhyasa.settings import val
import abhyasa.demo  # adds Learning.demo_for

## What a talk is

A talk is a document on the learning shelf, of kind `talk`. It points at its walk in `src` and copies what it shows, so a change
to the talk never changes the walk. `minutes` is the slot of the presenter. `phones` is the choice "Room phones". `live` is the id
of the run that is open now, or None.

These names give the shape. A slide has one of `SLIDE_KINDS`, and a question has one of `ASK_KINDS`.

In [ ]:
#| export
TALK_SCHEMA = 1
SLIDE_KINDS = ('idea', 'pause', 'questions', 'recap', 'primer', 'board', 'timeline', 'code')
ASK_KINDS = ('choice', 'order', 'words', 'scale', 'none')
BEATS = ('passage', 'figure', 'demo', 'ask')
QUESTIONS_MIN = 5   # the minutes of a `questions` slide
RECAP_MIN = 2       # the minutes of a `recap` slide
WORDS_MOST = 80     # the most characters of a words answer

## Slides and beats

Each press of D shows one beat more. `passage`, `figure` and `demo` show a part of the slide. `ask` opens the question.
`lit:<text>` lights a text in the passage, and `lit:#<id>` lights a part of the figure or the demo. Leela adds `lines:a-b` and
`out` for its code slides.

A beat that shows a part needs that part on the slide. This table gives the part for each beat.

In [ ]:
#| export
_PART = dict(passage='passage', figure='figure', demo='demo', ask='question')

def _beat_ok(b) -> bool:
    "True when `b` is a beat that the stage knows."
    return isinstance(b, str) and (b in BEATS or b == 'out' or b.startswith('lit:') or bool(re.fullmatch(r'lines:\d+(-\d+)?', b)))

A figure is an image id, never a path or a URL: `asset:<path in the vault assets>`. An id with another scheme, a `..` part,
a leading `/`, a backslash, a second `:` or a control character is refused.

In [ ]:
#| export
def _image_ok(f, schemes=('asset',)) -> bool:
    "True when `f` is an image id with one of `schemes`."
    if not isinstance(f, str) or ':' not in f or any(ord(ch) < 32 or ord(ch) == 127 for ch in f): return False
    sch, p = f.split(':', 1)
    return sch in schemes and bool(p) and ':' not in p and not p.startswith('/') and '\\' not in p and '..' not in p.split('/')

A plain path and a path out of the assets folder are not image ids. A good id is.

In [ ]:
assert _image_ok('asset:attention/images/fig2.png')
assert not any(_image_ok(f) for f in ('/Users/k/fig.png', 'asset:../vault.db', 'asset:/etc/x.png', 'ws:a.png', 'asset:a:b.png', 'asset:a\\b.png', 'asset:a\nb.png', None))

`_ask_problems` checks the question of one slide. A choice needs 2 to 6 options and the index of one of them. An order
needs the indexes of all its options, each one time.

In [ ]:
#| export
def _ask_problems(sid, a) -> list:
    "The problems of the question `a` of slide `sid`."
    if a is None: return []
    k = a.get('kind') if isinstance(a, dict) else a
    if not isinstance(a, dict) or k not in ASK_KINDS: return [f'{sid}: unknown ask kind {k!r}']
    opts, ans = a.get('options'), a.get('answer')
    is_int = lambda x: isinstance(x, int) and not isinstance(x, bool)
    if k == 'choice':
        if not isinstance(opts, list) or not 2 <= len(opts) <= 6: return [f'{sid}: a choice needs 2 to 6 options']
        if not (is_int(ans) and 0 <= ans < len(opts)): return [f'{sid}: the answer {ans!r} is not an option']
    if k == 'order':
        n = len(opts) if isinstance(opts, list) else 0
        if not (isinstance(ans, list) and all(map(is_int, ans)) and sorted(ans) == list(range(n))): return [f'{sid}: the answer {ans!r} is not an order of the options']
    return []

`_beat_problems` checks the beats of one slide against what the slide shows.

In [ ]:
#| export
def _beat_problems(sid, s:dict, show:dict) -> list:
    "The problems of the beats of slide `s`."
    out, beats = [], s.get('beats') or []
    for b in beats if isinstance(beats, list) else [beats]:
        if not _beat_ok(b): out.append(f'{sid}: unknown beat {b!r}'); continue
        if b in _PART and not (s.get('ask') if b == 'ask' else show.get(b)): out.append(f'{sid}: the beat {b!r} needs a {_PART[b]} to show')
        if b.startswith('lit:') and not b.startswith('lit:#'):
            text, passage = b[4:], show.get('passage')
            if not (text and isinstance(passage, str) and text in passage): out.append(f'{sid}: the beat {b!r} lights text that is not in the passage')
    return out

`check_talk` gives the problems of a talk, slide by slide in slide order. One slide can give more than one problem. A good talk
gives `[]`. It never raises, so a host can show the problems of a talk from a file.

In [ ]:
#| export
def check_talk(t:dict, schemes=('asset',)) -> list:
    "The problems of talk `t`; `[]` when the talk is good. It never raises."
    if not isinstance(t, dict): return ['the talk is not a dict']
    out = [] if (v := t.get('schema')) == TALK_SCHEMA else [f'schema {v} is not 1']
    slides, seen = t.get('slides'), set()
    if not isinstance(slides, list): return out + ['the slides are not a list']
    for s in slides:
        if not isinstance(s, dict): out.append('a slide is not a dict'); continue
        sid = s.get('id')
        try: twice = sid in seen; seen.add(sid)
        except TypeError: twice = False
        if twice: out.append(f'{sid}: the slide id is used twice')
        if (k := s.get('kind')) not in SLIDE_KINDS: out.append(f'{sid}: unknown kind {k!r}')
        out += _ask_problems(sid, s.get('ask'))
        show = s.get('show') if isinstance(s.get('show'), dict) else {}
        out += _beat_problems(sid, s, show)
        if (f := show.get('figure')) is not None and not _image_ok(f, schemes): out.append(f'{sid}: the figure {f!r} is not an image id')
    return out

`TALK_CASES` holds a good talk and one changed copy of it for each problem, with the problems that each must give. Leela runs
the same cases on its own copy of the checks.

In [ ]:
#| export
def _good() -> dict:
    "A small good talk with one slide."
    return dict(id='t-1', kind='talk', schema=1, title='T', src={'walk': 'w-1'}, minutes=10, phones=False, live=None,
                slides=[dict(id='s1', kind='idea', title='A', section='', notes='', mins=3,
                             show=dict(passage='Scores become weights.', figure='asset:a/fig.png', demo=None),
                             beats=['passage', 'lit:weights', 'figure'], ask=dict(kind='choice', q='Q?', options=['a', 'b'], answer=1))])

def _case(talk:dict=None, **slide) -> dict:
    "The good talk with the keys of `talk` and the keys of `slide` on its first slide changed."
    t = _good(); t.update(talk or {}); t['slides'][0].update(slide)
    return t

def _with_show(**kw): return {**_good()['slides'][0]['show'], **kw}
def _with_ask(**kw): return {**_good()['slides'][0]['ask'], **kw}

TALK_CASES = [
    (_good(), []),
    (_case(dict(schema=2)), ['schema 2 is not 1']),
    (_case(kind='video'), ["s1: unknown kind 'video'"]),
    (_case(ask=_with_ask(answer=2)), ['s1: the answer 2 is not an option']),
    (_case(ask=_with_ask(options=['a'], answer=0)), ['s1: a choice needs 2 to 6 options']),
    (_case(beats=['passage', 'demo']), ["s1: the beat 'demo' needs a demo to show"]),
    (_case(beats=['passage', 'lit:gradients']), ["s1: the beat 'lit:gradients' lights text that is not in the passage"]),
    (_case(show=_with_show(figure='/Users/k/fig.png')), ["s1: the figure '/Users/k/fig.png' is not an image id"]),
    (_case(show=_with_show(figure='asset:../vault.db')), ["s1: the figure 'asset:../vault.db' is not an image id"]),
    (dict(_good(), slides=_good()['slides'] * 2), ['s1: the slide id is used twice']),
    (_case(ask=_with_ask(kind='poll')), ["s1: unknown ask kind 'poll'"]),
    (_case(beats=['passage', 'fly']), ["s1: unknown beat 'fly'"]),
    (_case(ask=dict(kind='order', q='Q?', options=['a', 'b', 'c'], answer=[0, 0, 1])), ['s1: the answer [0, 0, 1] is not an order of the options'])]

A talk that is not a dict, or a slide with odd values, gives problems and does not raise.

In [ ]:
assert check_talk(None) == ['the talk is not a dict'] and check_talk(dict(schema=1)) == ['the slides are not a list']
assert check_talk(dict(schema=1, slides=[dict(id=['x'], kind='idea', beats='fly', ask='poll', show=None)])) == \
    ["['x']: unknown ask kind 'poll'", "['x']: unknown beat 'fly'"]

## Questions for the room

A demo gate with fixed options can be a room question with no model call. `ask_from_gate` has the contract of the future
`rupaka.poll.of_gate`. The options are a list, or `{"from": path}` in the state of the spec. The answer is `{"is": value}`, or a
path in the state. A path that is not in the state (it needs `derive`) gives None, and so do fewer than 2 or more than 6 options.
The prompt is drawn with the values of the state. A `{name}` with no value stays as it is.

In [ ]:
#| export
_NONE = object()

def _at(state, path:str):
    "The value at the dotted `path` in `state`, or `_NONE`."
    v = state
    for k in str(path).split('.'):
        if isinstance(v, dict) and k in v: v = v[k]
        elif isinstance(v, list) and k.isdigit() and int(k) < len(v): v = v[int(k)]
        else: return _NONE
    return v

def _gates(bs):
    "The gate blocks in `bs`, nested blocks too, in order."
    for b in bs or []:
        if not isinstance(b, dict): continue
        if b.get('type') == 'gate': yield b
        yield from _gates(b.get('blocks'))

`ask_from_gate` gives a choice from gate `gate` of `spec`, or None.

In [ ]:
#| export
def ask_from_gate(spec:dict, gate:str) -> dict|None:
    "A choice question from the gate `gate` of `spec`, with the index of the right option; None when the gate can not be a room question."
    g, state = first(b for b in _gates((spec or {}).get('blocks')) if b.get('id') == gate), (spec or {}).get('state') or {}
    if g is None: return None
    opts = g.get('options')
    if isinstance(opts, dict): opts = _at(state, opts.get('from', ''))
    ans = g.get('answer')
    ans = ans['is'] if isinstance(ans, dict) and 'is' in ans else _at(state, ans) if isinstance(ans, str) else _NONE
    if not isinstance(opts, list) or not 2 <= len(opts) <= 6 or ans is _NONE or ans not in opts: return None
    q = re.sub(r'\{(\w+)\}', lambda m: str(state[m[1]]) if m[1] in state else m[0], g.get('prompt') or '')
    return dict(kind='choice', q=q, options=list(opts), answer=opts.index(ans)) if q else None

`public_ask` keeps only what the room can see: the kind, the question, the options, and the most characters of a words answer.

In [ ]:
#| export
def public_ask(ask:dict|None) -> dict|None:
    "The part of `ask` that the room sees: no answer and no source."
    return None if ask is None else {k: ask[k] for k in ('kind', 'q', 'options', 'most') if k in ask}

A short check becomes a choice of 4 options through one model call, with the prompt `talk.choice`. The answer stays in the
talk. When the call fails, or the reply is bad, the question is a words question.

In [ ]:
#| export
CHOICE_SP = register('talk.choice', 'Talk question', 'How the tutor turns a short check into a choice question for a room.', """You turn one check into a choice question. A presenter shows the question to a room, and each person picks one option.
- Ask about the idea that the statement and the passage give. Ask one thing only.
- Write a short question that a person can read in a few seconds.
- Write 4 short options. One option is right. Each other option is a mistake that a learner can make.
- Do not use "all of the above" or "none of the above". Do not put the answer in the question.
Return JSON: {"q": "the question", "options": ["4 options"], "answer": the index of the right option, from 0}""")

`_choice` gives the reply as a choice when it has a question, 4 different options and the index of one of them, else None.

In [ ]:
#| export
def _choice(r) -> dict|None:
    "The reply `r` as a choice question, or None when it is bad."
    if not isinstance(r, dict): return None
    q, opts, ans = r.get('q'), r.get('options'), r.get('answer')
    good = (isinstance(q, str) and q.strip() and isinstance(opts, list) and len(opts) == 4 and all(isinstance(o, str) and o.strip() for o in opts)
            and len(set(opts)) == 4 and isinstance(ans, int) and not isinstance(ans, bool) and 0 <= ans < 4)
    return dict(kind='choice', q=q.strip(), options=[o.strip() for o in opts], answer=ans) if good else None

`ask_from_check` makes the call with the model task `talk`. `ask_for_concept` does the same with the statement of the idea as the passage.

In [ ]:
#| export
def ask_from_check(lr:Learning, ask, slug:str, passage:str='') -> dict:
    "A choice question on concept `slug` from one model call, or a words question when the call fails."
    c = lr.concept(slug) or AttrDict(name=slug, statement='')
    p = f"Idea: {c.get('name') or slug}\nStatement: {c.get('statement') or ''}\nPassage: {passage}"
    try: r = _choice(ask_json(for_task(ask, 'talk'), p, prompt('talk.choice')))
    except Exception: r = None
    return r or dict(kind='words', q=f"Say in one or two sentences: what is {c.get('name') or slug}?", most=WORDS_MOST)

def ask_for_concept(lr:Learning, ask, slug:str) -> dict:
    "`ask_from_check` with the statement of concept `slug` as the passage."
    return ask_from_check(lr, ask, slug, (lr.concept(slug) or {}).get('statement') or '')

## Minutes

A slide with a question takes the minutes of its walk step plus `talks.ask_min`. The talk takes the sum of its slides.

In [ ]:
#| export
def slide_minutes(step:dict, ask:dict|None) -> int:
    "The minutes of the slide of walk step `step` with the question `ask`."
    return minutes(step) + (val('talks.ask_min') if ask and ask.get('kind') != 'none' else 0)

def talk_minutes(t) -> int:
    "The minutes of all the slides of talk `t`."
    return sum(s.get('mins') or 0 for s in t['slides'])

## A talk from a walk

Each walk step that is not skipped gives one `idea` slide. A demo or an order check takes its question from the first gate of the
demo that can be a room question. A step with no demo, or with no such gate, uses its passage and a short check. A code check
gives a words question, because a talk never runs code.

`_idea_ask` gives the demo slug (or None) and the question of one step.

In [ ]:
#| export
def _idea_ask(lr:Learning, ask, s:dict, passage:str, choice:bool) -> tuple:
    "The demo slug and the question of walk step `s`."
    kind = (s.get('check') or {}).get('kind') or 'short'
    if kind in ('demo', 'order') and (d := lr.demo_for(s['slug'], 'order' if kind == 'order' else None)):
        for g in _gates(d.spec.get('blocks')):
            if (a := ask_from_gate(d.spec, g.get('id'))): return d.slug, dict(a, src=dict(demo=d.slug, gate=g['id']))
    c = lr.concept(s['slug']) or {}
    if kind == 'code' or not choice: a = dict(kind='words', q=f"Say in one or two sentences: what is {c.get('name') or s['slug']}?", most=WORDS_MOST)
    else: a = ask_from_check(lr, ask, s['slug'], passage)
    return None, dict(a, src=dict(check='code' if kind == 'code' else 'short'))

`_idea` makes the slide of one step. Its beats show the passage, then the figure, then the demo, each when the slide has it.

In [ ]:
#| export
def _idea(lr:Learning, ask, i:int, s:dict, sid:str, choice:bool) -> dict:
    "The `idea` slide `sid` of walk step `i`."
    r = s.get('read') or {}
    passage, fig = r.get('text') or '', r.get('figure')
    demo, a = _idea_ask(lr, ask, s, passage, choice)
    show = dict(passage=passage, figure=f'asset:{fig}' if fig else None, demo=demo)
    beats = [b for b in ('passage', 'figure', 'demo') if show[b]]
    c = lr.concept(s['slug']) or {}
    return dict(id=sid, kind='idea', step=i, concept=s['slug'], title=c.get('name') or s['slug'], section=r.get('section') or '',
                show=show, beats=beats, notes=s.get('note') or '', ask=a, mins=slide_minutes(s, a))

`talk_from_walk` makes the slides, then a `questions` slide and a `recap` slide, and saves the talk. The slot is the estimate
rounded up to a multiple of 5 minutes. The id is made from the title, so it matches `^[a-z0-9-]{1,48}$`.

In [ ]:
#| export
def _end(sid:str, kind:str, title:str, mins:int) -> dict:
    "A closing slide with no step and no question."
    return dict(id=sid, kind=kind, title=title, section='', show=dict(passage='', figure=None, demo=None), beats=[], notes='', ask=None, mins=mins)

def _talk_id(title:str) -> str:
    "A new talk id from `title`."
    return re.sub(r'[^a-z0-9-]', '', f'{slugify(title)[:34]}-talk-{uuid.uuid4().hex[:4]}')[:48]

def talk_from_walk(lr:Learning, ask, jid:str, choice:bool|None=None) -> AttrDict:
    "Make a talk from walk `jid` and save it. `choice` (default `talks.choice`) makes a choice question from each short check."
    j = lr.journey(jid)
    if j is None: raise KeyError(f'no journey {jid!r}')
    choice = val('talks.choice') if choice is None else choice
    steps = [(i, s) for i, s in enumerate(j.steps) if s.get('status') != 'skip']
    slides = [_idea(lr, ask, i, s, f's{k+1}', choice) for k, (i, s) in enumerate(steps)]
    n = len(slides)
    slides += [_end(f's{n+1}', 'questions', 'Questions', QUESTIONS_MIN), _end(f's{n+2}', 'recap', 'What we covered', RECAP_MIN)]
    t = dict(id=_talk_id(j.title), kind='talk', schema=TALK_SCHEMA, title=j.title, src={'walk': jid}, made=time.time(),
             minutes=0, phones=False, live=None, slides=slides)
    t['minutes'] = 5 * math.ceil(talk_minutes(t) / 5)
    return lr.put_talk(t)

## Store calls

A talk is a document of kind `talk`. `put_talk` refuses a talk with problems, and replaces an earlier talk with the same id.

In [ ]:
#| export
def _tmeta(r): return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']})

@patch
def put_talk(self:Learning, t:dict) -> AttrDict:
    "Save talk `t`, and replace an earlier one with the same id. A talk with problems raises `ValueError`."
    t = {k: v for k, v in dict(t).items() if k != 'doc_id'}
    if (ps := check_talk(t)): raise ValueError('; '.join(ps))
    t['updated'] = time.time()
    text = f"Talk: {t['title']} ({t['id']})\n\nSlides: {', '.join(s.get('title') or '' for s in t['slides'])}"
    r = self.v.add(text, title=f"Talk: {t['title']}", source=_src('talk', t['id']), kind='talk', meta=t)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **t)
    return self.talk(t['id'])

`talk` gives one talk by id, `talks` gives all of them with the newest first, and `drop_talk` removes one.

In [ ]:
#| export
@patch
def talk(self:Learning, tid:str) -> AttrDict|None:
    "One talk by id."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('talk', tid)]))
    return _tmeta(r) if r else None

@patch
def talks(self:Learning) -> L:
    "All talks, the most recently changed first."
    return L(sorted(map(_tmeta, self._docs('talk')), key=lambda t: -t.get('updated', 0)))

@patch
def drop_talk(self:Learning, tid:str):
    "Remove a talk. Its walk stays."
    if (t := self.talk(tid)): self.v.forget(t.doc_id)

## Tests

These tests use the small paper in `tests/walk_data.py`, the walk of `tests/talk_data.py` and a fake model.

In [ ]:
#| hide
import sys, abhyasa
from pathlib import Path
sys.path.insert(0, str(Path(abhyasa.__file__).parents[1]/'tests'))
from talk_data import *
from abhyasa.store import temp_learning
from abhyasa.llm import Fake

A talk from the walk: one slide for each idea, then the questions and the recap.

In [ ]:
lr = temp_learning(); jid = talk_walk(lr); ask = Fake(TALK_RULES)
t = talk_from_walk(lr, ask, jid)
s1, s2, s3 = t.slides[:3]
assert [s['id'] for s in t.slides] == ['s1', 's2', 's3', 's4', 's5']
assert [s['kind'] for s in t.slides] == ['idea', 'idea', 'idea', 'questions', 'recap']

The first slide has the title, the section, the step and the concept of its idea. It shows the passage, and its question is a choice from the short check.

In [ ]:
assert (s1['title'], s1['section'], s1['step'], s1['concept']) == ('Sequential computation', '1 Introduction', 0, 'sequential-computation')
assert s1['show']['passage'].startswith('This inherently sequential nature') and s1['show']['figure'] is None and s1['beats'] == ['passage']
assert s1['ask'] == dict(kind='choice', **CHOICE, src=dict(check='short'))

The second slide shows the figure as an image id, and its notes are the note of the step. The third slide shows the demo, and its question comes from the gate `hot`.

In [ ]:
assert s2['show']['figure'] == 'asset:attention/images/fig2.png' and s2['beats'] == ['passage', 'figure'] and s2['notes'] == 'Softmax makes weights from scores.'
assert s3['show']['demo'] == 'softmax-temperature' and s3['beats'] == ['passage', 'demo']
assert s3['ask'] == dict(kind='choice', q='T goes from 1 to 2. What happens to the share of Paris?',
                         options=['It rises', 'It falls', 'It stays the same'], answer=1, src=dict(demo='softmax-temperature', gate='hot'))

The minutes add to 20. The demo step makes no model call. The talk is saved, and it passes its checks.

In [ ]:
assert [s['mins'] for s in t.slides] == [4, 4, 5, 5, 2] and talk_minutes(t) == 20 and t.minutes == 20
assert len([c for c in ask.calls if 'You turn one check into a choice question' in c[0] + c[1]]) == 2   # no call for the demo step
assert t.src == {'walk': jid} and t.schema == TALK_SCHEMA and t.live is None and t.phones is False
assert check_talk(t) == [] and lr.talk(t.id).title == 'Attention Is All You Need' and [x.id for x in lr.talks()] == [t.id]

A bad reply, or `choice=False`, gives a words question.

In [ ]:
lr2 = temp_learning(); jid2 = talk_walk(lr2)
bad = talk_from_walk(lr2, Fake({**TALK_RULES, 'You turn one check into a choice question': dict(q='', options=[], answer=9)}), jid2)
assert bad.slides[0]['ask'] == dict(kind='words', q='Say in one or two sentences: what is Sequential computation?', most=80, src=dict(check='short'))
off = talk_from_walk(lr2, Fake(TALK_RULES), jid2, choice=False)
assert off.slides[1]['ask']['kind'] == 'words'

A demo step with no demo falls back to its idea: no demo on the slide, and a choice from a short check.

In [ ]:
lr2.v.forget(lr2.demo('softmax-temperature').doc_id)
nodemo = talk_from_walk(lr2, Fake(TALK_RULES), jid2)
assert nodemo.slides[2]['show']['demo'] is None and nodemo.slides[2]['beats'] == ['passage'] and nodemo.slides[2]['ask']['kind'] == 'choice'

The talk id matches the form of an id, also for a long title. `drop_talk` removes a talk.

In [ ]:
assert all(re.fullmatch(r'[a-z0-9-]{1,48}', x.id) for x in lr2.talks()) and re.fullmatch(r'[a-z0-9-]{1,48}', _talk_id('Ä very long title ' * 9))
lr2.drop_talk(bad.id)
assert lr2.talk(bad.id) is None and bad.id not in [x.id for x in lr2.talks()]

A question from a gate needs fixed options and an answer that the spec holds.

In [ ]:
g = gallery()
assert ask_from_gate(g['parabola-vertex'], 'move') == dict(kind='choice', q='b goes from 0 to 2. Where does the lowest point move?',
                                                          options=['Left and down', 'Right and down', 'Straight up', 'It stays'], answer=0)
assert ask_from_gate(g['reciprocal-rank-fusion'], 'win') is None   # the answer needs derive
assert ask_from_gate(g['vowel-sandhi'], 'join') is None            # the options need derive
assert ask_from_gate(g['softmax-temperature'], 'nope') is None

Options and an answer from a state path give a choice, and the prompt gets the values of the state. A `{name}` with no value stays.

In [ ]:
spec = dict(state=dict(k=60, opts=['x', 'y'], best='y'), blocks=[dict(type='row', blocks=[dict(type='gate', id='g', prompt='With k = {k} and {m}?',
                                                                                         options={'from': 'opts'}, answer='best')])])
assert ask_from_gate(spec, 'g') == dict(kind='choice', q='With k = 60 and {m}?', options=['x', 'y'], answer=1)

The public part of a question has no answer and no source.

In [ ]:
assert public_ask(s3['ask']) == dict(kind='choice', q='T goes from 1 to 2. What happens to the share of Paris?', options=['It rises', 'It falls', 'It stays the same'])
assert public_ask(None) is None

`TALK_CASES`: each talk gives exactly its problems, and `put_talk` refuses a talk with a problem.

In [ ]:
GOOD = dict(id='t-1', kind='talk', schema=1, title='T', src={'walk': 'w-1'}, minutes=10, phones=False, live=None,
            slides=[dict(id='s1', kind='idea', title='A', section='', notes='', mins=3,
                         show=dict(passage='Scores become weights.', figure='asset:a/fig.png', demo=None),
                         beats=['passage', 'lit:weights', 'figure'], ask=dict(kind='choice', q='Q?', options=['a', 'b'], answer=1))])
for case, want in TALK_CASES: assert check_talk(case) == want, (case, want)
wants = [w for _, w in TALK_CASES]
assert [] in wants and ['schema 2 is not 1'] in wants and ["s1: unknown kind 'video'"] in wants
assert ['s1: the answer 2 is not an option'] in wants and ['s1: a choice needs 2 to 6 options'] in wants
assert ["s1: the beat 'demo' needs a demo to show"] in wants and ["s1: the beat 'lit:gradients' lights text that is not in the passage"] in wants
assert ["s1: the figure '/Users/k/fig.png' is not an image id"] in wants and ["s1: the figure 'asset:../vault.db' is not an image id"] in wants
assert ['s1: the slide id is used twice'] in wants and ["s1: unknown ask kind 'poll'"] in wants and ["s1: unknown beat 'fly'"] in wants
assert ['s1: the answer [0, 0, 1] is not an order of the options'] in wants
test_fail(lambda: lr.put_talk(dict(GOOD, schema=2)), contains='schema 2 is not 1')

The good talk of the cases is the same as `GOOD`, and there is one case for each problem.

In [ ]:
assert TALK_CASES[0][0] == GOOD and len(TALK_CASES) == 13

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()